# 06 - Train 3 Models

### Notebook Overview

Train three regression models manually using scikit-learn, starting with a simple baseline and progressing to more complex approaches. Each model is cross-validated and tuned.

**Input:** `../data/x-train.csv`, `../data/y-train.csv`  
**Output:** Trained models saved to `../models/`

**Models:**
1. Linear Regression (baseline)
2. Random Forest
3. Gradient Boosting

### 1 - Imports

In [1]:
# Core Libraries
import pandas as pd
import numpy as np
import joblib
import os

# Model building
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score

### 2 - Load Data

In [2]:
X_train = pd.read_csv("../data/x-train.csv")
y_train = pd.read_csv("../data/y-train.csv").squeeze()

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")

X_train: (164, 35)
y_train: (164,)


In [3]:
os.makedirs("../models", exist_ok=True)

### 3 - Cross-Validation Strategy

With only 164 training rows, we can't afford a separate validation set. Instead we use 5-fold cross-validation: the training data is split into 5 parts, and each model is trained 5 times, each time holding out a different part for validation. The average score across all 5 folds gives a more reliable estimate of performance than a single train/validation split.

We use **negative mean absolute error (neg_MAE)** for cross-validation. MAE is intuitive for price prediction because it tells us the average dollar amount we're off by. scikit-learn negates it (so higher is better), but we'll flip it back for readability.

In [4]:
def evaluate_cv(model, X, y, name):
    """Run 5-fold CV and print results."""
    mae_scores = -cross_val_score(model, X, y, cv=5, scoring="neg_mean_absolute_error")
    r2_scores = cross_val_score(model, X, y, cv=5, scoring="r2")
    
    print(f"\n{name}")
    print(f"  MAE:  ${mae_scores.mean():,.0f} (+/- ${mae_scores.std():,.0f})")
    print(f"  R2:   {r2_scores.mean():.3f} (+/- {r2_scores.std():.3f})")
    
    return {"model": name, "mae_mean": mae_scores.mean(), "mae_std": mae_scores.std(),
            "r2_mean": r2_scores.mean(), "r2_std": r2_scores.std()}

### 4 - Model 1: Linear Regression (Baseline)

Linear regression is the simplest approach: it finds the best straight-line (or hyperplane) relationship between features and price. It assumes the relationship is linear, which is rarely perfectly true, but it gives us a sensible floor to beat.

If the more complex models can't meaningfully beat this, it means either the data is inherently linear or we don't have enough data for complex models to shine.

In [5]:
lr = LinearRegression()
lr_results = evaluate_cv(lr, X_train, y_train, "Linear Regression")

# Fit on full training set and save
lr.fit(X_train, y_train)
joblib.dump(lr, "../models/linear-regression.pkl")
print("  Saved to models/linear-regression.pkl")


Linear Regression
  MAE:  $2,034 (+/- $175)
  R2:   0.843 (+/- 0.028)
  Saved to models/linear-regression.pkl


### 5 - Model 2: Random Forest

A Random Forest builds many decision trees in parallel, each trained on a random subset of the data and features. The final prediction is the average of all trees. This "wisdom of crowds" approach reduces overfitting compared to a single tree.

Key parameters:
- `n_estimators`: number of trees. More trees = more stable predictions, but diminishing returns past ~200.
- `max_depth`: how deep each tree can grow. Deeper trees fit more complex patterns but risk overfitting.
- `min_samples_leaf`: minimum data points in each leaf node. Higher values prevent the tree from learning noise.

In [6]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
)
rf_results = evaluate_cv(rf, X_train, y_train, "Random Forest")

rf.fit(X_train, y_train)
joblib.dump(rf, "../models/random-forest.pkl")
print("  Saved to models/random-forest.pkl")


Random Forest
  MAE:  $1,693 (+/- $327)
  R2:   0.879 (+/- 0.062)
  Saved to models/random-forest.pkl


### 6 - Model 3: Gradient Boosting

Gradient Boosting also builds trees, but sequentially rather than in parallel. Each new tree focuses on correcting the errors of the previous ones. Think of it as a team where each member specialises in fixing what the previous member got wrong.

This usually produces the most accurate predictions, but is more prone to overfitting than Random Forest. Key parameters:
- `n_estimators`: number of boosting rounds.
- `learning_rate`: how much each new tree contributes. Lower values need more trees but generalise better.
- `max_depth`: kept shallow (3-5) because each tree is meant to be a "weak learner".

In [7]:
gb = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    min_samples_leaf=5,
    random_state=42,
)
gb_results = evaluate_cv(gb, X_train, y_train, "Gradient Boosting")

gb.fit(X_train, y_train)
joblib.dump(gb, "../models/gradient-boosting.pkl")
print("  Saved to models/gradient-boosting.pkl")


Gradient Boosting
  MAE:  $1,553 (+/- $310)
  R2:   0.900 (+/- 0.056)
  Saved to models/gradient-boosting.pkl


### 7 - Cross-Validation Comparison

In [8]:
results = pd.DataFrame([lr_results, rf_results, gb_results])
results["mae_display"] = results["mae_mean"].apply(lambda x: f"${x:,.0f}") + " (+/- " + results["mae_std"].apply(lambda x: f"${x:,.0f}") + ")"
results["r2_display"] = results["r2_mean"].apply(lambda x: f"{x:.3f}") + " (+/- " + results["r2_std"].apply(lambda x: f"{x:.3f}") + ")"

print("Cross-Validation Results (5-fold):")
print(results[["model", "mae_display", "r2_display"]].to_string(index=False))

Cross-Validation Results (5-fold):
            model       mae_display        r2_display
Linear Regression $2,034 (+/- $175) 0.843 (+/- 0.028)
    Random Forest $1,693 (+/- $327) 0.879 (+/- 0.062)
Gradient Boosting $1,553 (+/- $310) 0.900 (+/- 0.056)


### 8 - Summary

**Cross-validation results (5-fold):**

| Model | MAE | R2 |
|-------|-----|-----|
| Linear Regression | $2,034 (+/- $175) | 0.843 (+/- 0.028) |
| Random Forest | $1,693 (+/- $327) | 0.879 (+/- 0.062) |
| Gradient Boosting | $1,553 (+/- $310) | 0.900 (+/- 0.056) |

**Interpretation:**
- All three models perform reasonably well. An R2 of 0.90 means Gradient Boosting explains 90% of the variance in car prices.
- Gradient Boosting leads on both metrics, with an average prediction error of ~$1,553 on a dataset where prices range from $5k to $45k.
- Linear Regression is the most stable (lowest variance across folds) but least accurate. This is typical: simpler models are more consistent but have a lower ceiling.
- The tree-based models show higher fold-to-fold variance, which is expected with a small dataset (only ~33 samples per fold).

All models saved to `models/` for evaluation in the next notebook.